In [2]:
# ============================================================
# KHỞI TẠO: Import module & nạp dữ liệu GCS
# ============================================================
# data_quality.py nằm cùng thư mục (data/quality_checks/).

import pandas as pd
import numpy as np

from data_quality import (
    audit,
    check_overview,
    check_plausibility,
    check_distribution,
    check_hidden_issues,
    check_logic,
)

# Đường dẫn (tính từ data/quality_checks/)
gcs_input    = "../processed/gcs_clean.csv"
patient_input = "../processed/patient_clean.csv"

df_gcs = pd.read_csv(gcs_input)
df_patient = pd.read_csv(patient_input)

print("Đã nạp gcs_clean.csv    :", df_gcs.shape)
print("Đã nạp patient_clean.csv:", df_patient.shape)
print("\n5 dòng đầu tiên của gcs:")
print(df_gcs.head())

Đã nạp gcs_clean.csv    : (2074, 2)
Đã nạp patient_clean.csv: (2326, 5)

5 dòng đầu tiên của gcs:
   patientunitstayid   gcs
0             141765  15.0
1             143870  15.0
2             144815  15.0
3             145427  15.0
4             147307  15.0


In [3]:
# ============================================================
# KIỂM TRA CHUNG: dùng module data_quality
# ============================================================
# audit() chạy tất cả kiểm tra tổng quát.
# Với GCS: không có cột nhãn mortality trong file này,
# nên truyền label_col=None để bỏ qua phần mất cân bằng nhãn.
# Ngưỡng plausibility cho GCS: eyes 1–4, motor 1–6, verbal 1–5.

GCS_RULES = {
    "gcs": (3, 15, "điểm Glasgow"),
}

audit(
    df_gcs,
    name="gcs_clean.csv",
    label_col=None,                 # không có nhãn trong file GCS
    plausibility_rules=GCS_RULES,
)


TỔNG QUAN: gcs_clean.csv
Kích thước : 2,074 dòng × 2 cột
Bộ nhớ     : 32.5 KB

Kiểu dữ liệu:
patientunitstayid      int64
gcs                  float64

Missing values:
     missing   pct
gcs       51  2.46

Số dòng trùng lặp hoàn toàn: 0

KIỂM TRA PLAUSIBILITY (ngưỡng sinh lý)
✅ gcs             | min=      3.00 max=     15.00 | ngoài [3, 15] (điểm Glasgow): 0 dòng

PHÂN BỐ CÁC CỘT SỐ

--- patientunitstayid ---
  count=2,074  mean=1770066.03  std=1013468.57
  min=141765.00  q25=943763.00  median=1716234.50  q75=2747320.00  max=3353113.00

--- gcs ---
  count=2,023  mean=13.44  std=3.06
  min=3.00  q25=14.00  median=15.00  q75=15.00  max=15.00
  ⚠️ Phân bố lệch mạnh (skew=-2.21)

KIỂM TRA VẤN ĐỀ ẨN

KIỂM TRA LOGIC GIỮA CÁC CỘT
✅ patientunitstayid <= 0: 0 dòng

✅ HOÀN TẤT KIỂM TRA: gcs_clean.csv



In [4]:
# ============================================================
# KIỂM TRA RIÊNG: Khóa chính
# ============================================================
# Mỗi patientunitstayid chỉ được xuất hiện đúng 1 lần.

print("=" * 60)
print("KIỂM TRA RIÊNG: KHÓA CHÍNH")
print("=" * 60)

n_rows   = len(df_gcs)
n_unique = df_gcs["patientunitstayid"].nunique()
n_dup    = n_rows - n_unique

print(f"\nSố dòng              : {n_rows:,}")
print(f"Số patientunitstayid : {n_unique:,}")
print(f"Số trùng lặp         : {n_dup}")
print(f"{'✅' if n_dup == 0 else '⚠️'} Khóa chính duy nhất")

n_bad_id = (df_gcs["patientunitstayid"] <= 0).sum()
print(f"{'✅' if n_bad_id == 0 else '⚠️'} patientunitstayid <= 0: {n_bad_id} dòng")

KIỂM TRA RIÊNG: KHÓA CHÍNH

Số dòng              : 2,074
Số patientunitstayid : 2,074
Số trùng lặp         : 0
✅ Khóa chính duy nhất
✅ patientunitstayid <= 0: 0 dòng


In [5]:
# ============================================================
# KIỂM TRA RIÊNG: Phạm vi & phân bố GCS
# ============================================================
# GCS hợp lệ: 3–15. Giá trị ngoài khoảng này là lỗi
# (thường do sentinel -1 chưa được xử lý, hoặc cộng sai).

print("=" * 60)
print("KIỂM TRA RIÊNG: PHẠM VI & PHÂN BỐ GCS")
print("=" * 60)

gcs = df_gcs["gcs"].dropna()

print(f"\ncount  = {len(gcs):,} / {len(df_gcs):,} ({len(gcs)/len(df_gcs)*100:.2f}%)")
print(f"mean   = {gcs.mean():.2f}")
print(f"std    = {gcs.std():.2f}")
print(f"min    = {gcs.min():.0f}")
print(f"q25    = {gcs.quantile(.25):.0f}")
print(f"median = {gcs.median():.0f}")
print(f"q75    = {gcs.quantile(.75):.0f}")
print(f"max    = {gcs.max():.0f}")

# Kiểm tra phạm vi
n_bad_range = ((gcs < 3) | (gcs > 15)).sum()
print(f"\n{'✅' if n_bad_range == 0 else '⚠️'} GCS ngoài [3, 15]: {n_bad_range} dòng")

# Kiểm tra giá trị không nguyên (GCS phải là số nguyên)
n_non_int = (gcs % 1 != 0).sum()
print(f"{'✅' if n_non_int == 0 else '⚠️'} GCS không nguyên: {n_non_int} dòng")

# Phân bố
print("\nPhân bố GCS (value_counts, sắp xếp theo điểm):")
print(gcs.value_counts().sort_index().to_string())

# Skewness
skew = gcs.skew()
print(f"\nSkewness = {skew:.2f}")
if skew < -1:
    print("→ ℹ️ Phân bố lệch trái mạnh (đa số bệnh nhân tỉnh táo GCS 14–15)")
    print("   Đây là bình thường với dữ liệu GCS — XGBoost không cần transform.")

KIỂM TRA RIÊNG: PHẠM VI & PHÂN BỐ GCS

count  = 2,023 / 2,074 (97.54%)
mean   = 13.44
std    = 3.06
min    = 3
q25    = 14
median = 15
q75    = 15
max    = 15

✅ GCS ngoài [3, 15]: 0 dòng
✅ GCS không nguyên: 0 dòng

Phân bố GCS (value_counts, sắp xếp theo điểm):
gcs
3.0       81
4.0        3
5.0        4
6.0       34
7.0       37
8.0       41
9.0       39
10.0      37
11.0      53
12.0      59
13.0     103
14.0     210
15.0    1322

Skewness = -2.21
→ ℹ️ Phân bố lệch trái mạnh (đa số bệnh nhân tỉnh táo GCS 14–15)
   Đây là bình thường với dữ liệu GCS — XGBoost không cần transform.


In [6]:
# ============================================================
# KIỂM TRA RIÊNG: Tỷ lệ thiếu GCS
# ============================================================
# GCS có thể thiếu vì:
#   - Bệnh nhân không có trong apacheApsVar (252 người)
#   - Một trong 3 thành phần bị -1 (sentinel missing)

print("=" * 60)
print("KIỂM TRA RIÊNG: TỶ LỆ THIẾU GCS")
print("=" * 60)

n_missing = df_gcs["gcs"].isna().sum()
n_total   = len(df_gcs)
pct       = n_missing / n_total * 100

print(f"\nTổng số dòng      : {n_total:,}")
print(f"Số dòng thiếu GCS : {n_missing:,} ({pct:.2f}%)")

if pct < 5:
    print("✅ Tỷ lệ thiếu < 5% — chấp nhận được, giữ NaN cho XGBoost.")
elif pct < 15:
    print("ℹ️ Tỷ lệ thiếu 5–15% — vẫn giữ NaN, nhưng nên kiểm tra nhóm thiếu.")
else:
    print("⚠️ Tỷ lệ thiếu > 15% — cần phân tích kỹ nhóm thiếu GCS.")

KIỂM TRA RIÊNG: TỶ LỆ THIẾU GCS

Tổng số dòng      : 2,074
Số dòng thiếu GCS : 51 (2.46%)
✅ Tỷ lệ thiếu < 5% — chấp nhận được, giữ NaN cho XGBoost.


In [7]:
# ============================================================
# KIỂM TRA RIÊNG: So sánh cohort — ai thiếu GCS?
# ============================================================
# Merge patient_clean với gcs_clean để xem:
#   - Bao nhiêu bệnh nhân trong cohort KHÔNG có trong apache
#   - Nhóm thiếu GCS có tỷ lệ tử vong khác biệt không

print("=" * 60)
print("KIỂM TRA RIÊNG: SO SÁNH COHORT")
print("=" * 60)

df_check = df_patient.merge(df_gcs, on="patientunitstayid", how="left")
df_check["gcs_missing"] = df_check["gcs"].isna()

print(f"\nSố bệnh nhân trong cohort     : {len(df_patient):,}")
print(f"Số bệnh nhân có trong apache  : {(~df_check['gcs_missing']).sum():,}")
print(f"Số bệnh nhân THIẾU hoàn toàn  : {df_check['gcs_missing'].sum():,}")

# So sánh tỷ lệ tử vong
print("\nSo sánh tỷ lệ tử vong: có GCS vs thiếu GCS")
result = df_check.groupby("gcs_missing")["mortality"].agg(
    count="count", deaths="sum", mortality_rate="mean"
).round(4)
result.index = ["Có GCS", "Thiếu GCS"]
print(result.to_string())

if len(result) == 2:
    diff = result.loc["Thiếu GCS", "mortality_rate"] - result.loc["Có GCS", "mortality_rate"]
    print(f"\nChênh lệch: {diff:+.4f} ({diff*100:+.2f} điểm %)")
    if abs(diff) > 0.02:
        print("→ ⚠️ Khác biệt đáng kể — thiếu GCS mang tín hiệu! GIỮ NaN.")
    else:
        print("→ Không khác biệt đáng kể — giữ NaN vẫn an toàn.")

KIỂM TRA RIÊNG: SO SÁNH COHORT

Số bệnh nhân trong cohort     : 2,326
Số bệnh nhân có trong apache  : 2,023
Số bệnh nhân THIẾU hoàn toàn  : 303

So sánh tỷ lệ tử vong: có GCS vs thiếu GCS
           count  deaths  mortality_rate
Có GCS      2023     168          0.0830
Thiếu GCS    303      23          0.0759

Chênh lệch: -0.0071 (-0.71 điểm %)
→ Không khác biệt đáng kể — giữ NaN vẫn an toàn.


In [8]:
# ============================================================
# KIỂM TRA RIÊNG: Tương quan GCS với mortality
# ============================================================
# Nếu GCS thấp liên quan đến tử vong cao -> đặc trưng có ích.

print("=" * 60)
print("KIỂM TRA RIÊNG: GCS THEO NHÓM TỬ VONG")
print("=" * 60)

print("\nThống kê GCS theo mortality:")
print(df_check.groupby("mortality")["gcs"].agg(
    count="count", mean="mean", median="median",
    min="min", max="max"
).round(2).to_string())

# So sánh trung bình
gcs_alive   = df_check.loc[df_check["mortality"] == 0, "gcs"].mean()
gcs_expired = df_check.loc[df_check["mortality"] == 1, "gcs"].mean()

print(f"\nGCS TB - Alive  : {gcs_alive:.2f}")
print(f"GCS TB - Expired: {gcs_expired:.2f}")
print(f"Chênh lệch      : {gcs_expired - gcs_alive:+.2f}")

if gcs_expired < gcs_alive:
    print("→ ✅ Nhóm tử vong có GCS TB THẤP hơn — GCS là đặc trưng có ích")
else:
    print("→ ⚠️ Nhóm tử vong không có GCS thấp hơn rõ rệt")

# Tương quan
corr = df_check[["gcs", "mortality"]].corr().iloc[0, 1]
print(f"\nTương quan Pearson(GCS, mortality) = {corr:.3f}")
print(f"   (âm = GCS thấp → tử vong cao, đúng kỳ vọng lâm sàng)")

KIỂM TRA RIÊNG: GCS THEO NHÓM TỬ VONG

Thống kê GCS theo mortality:
           count   mean  median  min   max
mortality                                 
0           1855  13.69    15.0  3.0  15.0
1            168  10.70    13.0  3.0  15.0

GCS TB - Alive  : 13.69
GCS TB - Expired: 10.70
Chênh lệch      : -2.98
→ ✅ Nhóm tử vong có GCS TB THẤP hơn — GCS là đặc trưng có ích

Tương quan Pearson(GCS, mortality) = -0.269
   (âm = GCS thấp → tử vong cao, đúng kỳ vọng lâm sàng)


In [9]:
# ============================================================
# KẾT LUẬN: Checklist chất lượng GCS
# ============================================================

print("=" * 60)
print("CHECKLIST CHẤT LƯỢNG: gcs_clean.csv")
print("=" * 60)

gcs_valid = df_gcs["gcs"].dropna()

checks = {
    "Khóa chính duy nhất":        df_gcs["patientunitstayid"].duplicated().sum() == 0,
    "Không có dòng trùng hoàn toàn": df_gcs.duplicated().sum() == 0,
    "patientunitstayid > 0":      (df_gcs["patientunitstayid"] > 0).all(),
    "GCS trong [3, 15]":          ((gcs_valid >= 3) & (gcs_valid <= 15)).all(),
    "GCS là số nguyên":           (gcs_valid % 1 == 0).all(),
    "Tỷ lệ thiếu GCS < 10%":      df_gcs["gcs"].isna().mean() < 0.10,
    "Có đủ 2 lớp mortality":      df_patient["mortality"].nunique() == 2,
    "Cohort khớp patient":        df_gcs["patientunitstayid"].isin(
                                      df_patient["patientunitstayid"]).all(),
}

for name, passed in checks.items():
    print(f"{'✅' if passed else '❌'} {name}")

n_pass  = sum(checks.values())
n_total = len(checks)
print(f"\nKết quả: {n_pass}/{n_total} kiểm tra đạt")

if n_pass == n_total:
    print("\n🎉 DỮ LIỆU GCS SẴN SÀNG cho bước merge!")
else:
    print(f"\n⚠️ Còn {n_total - n_pass} vấn đề cần xử lý.")

CHECKLIST CHẤT LƯỢNG: gcs_clean.csv
✅ Khóa chính duy nhất
✅ Không có dòng trùng hoàn toàn
✅ patientunitstayid > 0
✅ GCS trong [3, 15]
✅ GCS là số nguyên
✅ Tỷ lệ thiếu GCS < 10%
✅ Có đủ 2 lớp mortality
✅ Cohort khớp patient

Kết quả: 8/8 kiểm tra đạt

🎉 DỮ LIỆU GCS SẴN SÀNG cho bước merge!
